In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/all_seasons.csv')

# Keep only the columns that actually exist in your CSV
cols = ['player_name','season','age','player_height','player_weight',
        'pts','reb','ast','net_rating','usg_pct','ts_pct',
        'draft_round','gp','team_abbreviation']
df = df[cols].copy()

# Filter low game count instead of low minutes (gp = games played)
df = df[df['gp'] >= 20].copy()

# Drop missing values
df = df.dropna(subset=['net_rating','pts','reb','ast','usg_pct','ts_pct'])

# Clean draft_round
df['draft_round'] = df['draft_round'].replace('Undrafted', '3')
df['draft_round'] = pd.to_numeric(df['draft_round'], errors='coerce').fillna(3)

# Create engineered features
df['log_pts'] = np.log1p(df['pts'])
df['pts_usg'] = df['pts'] * df['usg_pct']

print(f"Dataset shape: {df.shape}")
print(f"\nnet_rating summary:")
print(df['net_rating'].describe().round(2))

Dataset shape: (10720, 16)

net_rating summary:
count    10720.00
mean        -1.09
std          6.44
min        -40.00
25%         -5.30
50%         -0.80
75%          3.20
max         19.50
Name: net_rating, dtype: float64


In [4]:
features_full = ['age','pts','reb','ast','usg_pct','ts_pct','draft_round']
features_domain = ['pts','ast','reb','usg_pct']
features_transformed = ['log_pts','ast','reb','usg_pct','pts_usg']
target = 'net_rating'

X_full = df[features_full]
X_domain = df[features_domain]
X_transformed = df[features_transformed]
y = df[target]

X_train_f, X_test_f, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
X_train_d, X_test_d, _, _ = train_test_split(X_domain, y, test_size=0.2, random_state=42)
X_train_t, X_test_t, _, _ = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

print("Split done. Training rows:", len(y_train))

Split done. Training rows: 8576


In [6]:
def fit_ols(X_train, y_train, model_name):
    """Fit OLS with statsmodels and print summary."""
    X_const = sm.add_constant(X_train)
    model = sm.OLS(y_train, X_const).fit()
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print('='*60)
    print(model.summary())
    return model

model1 = fit_ols(X_train_f, y_train, "MODEL 1 — Full Model")
model2 = fit_ols(X_train_d, y_train, "MODEL 2 — Domain-Driven Model")
model3 = fit_ols(X_train_t, y_train, "MODEL 3 — Transformed Model")


  MODEL 1 — Full Model
                            OLS Regression Results                            
Dep. Variable:             net_rating   R-squared:                       0.204
Model:                            OLS   Adj. R-squared:                  0.203
Method:                 Least Squares   F-statistic:                     313.0
Date:                Thu, 02 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:25:46   Log-Likelihood:                -27120.
No. Observations:                8576   AIC:                         5.426e+04
Df Residuals:                    8568   BIC:                         5.431e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -23.2165    

In [8]:
def evaluate_model(model, X_test, y_test, model_name):
    X_const = sm.add_constant(X_test)
    y_pred = model.predict(X_const)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    n = len(y_test)
    k = X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    return {'Model': model_name, 'RMSE': round(rmse, 3),
            'R²': round(r2, 3), 'Adj R²': round(adj_r2, 3),
            'AIC': round(model.aic, 1), 'BIC': round(model.bic, 1)}

results = pd.DataFrame([
    evaluate_model(model1, X_test_f, y_test, "Full Model"),
    evaluate_model(model2, X_test_d, y_test, "Domain-Driven"),
    evaluate_model(model3, X_test_t, y_test, "Transformed"),
])

print(results.to_string(index=False))

        Model  RMSE    R²  Adj R²     AIC     BIC
   Full Model 5.822 0.209   0.207 54256.6 54313.1
Domain-Driven 6.044 0.148   0.147 55070.7 55106.0
  Transformed 6.016 0.156   0.154 55033.4 55075.7


## Model Comparison Table — Observations

| Model         | RMSE  | R²    | Adj R² | AIC     | BIC     |
|---------------|-------|-------|--------|---------|---------|
| Full Model    | 5.822 | 0.209 | 0.207  | 54256.6 | 54313.1 |
| Domain-Driven | 6.044 | 0.148 | 0.147  | 55070.7 | 55106.0 |
| Transformed   | 6.016 | 0.156 | 0.154  | 55033.4 | 55075.7 |

**Reading the table:**

- **RMSE (Root Mean Squared Error):** Measures average prediction error in net_rating points. Lower is better. Full Model wins here with RMSE = 5.822, meaning its predictions are off by ~5.8 net_rating points on average.
- **R² / Adj R²:** Percentage of variation in net_rating explained by the model. Full Model explains 20.9%, Transformed explains 15.6%, Domain-Driven explains 14.8%. Adjusted R² penalises extra predictors — Full Model still leads.
- **AIC / BIC:** Model fit score that penalises complexity. Lower is better. Full Model has the lowest AIC (54256.6), followed by Transformed (55033.4), then Domain-Driven (55070.7).

**Key observations:**
- Full Model leads on every metric — lowest RMSE, highest R², lowest AIC.
- Transformed Model beats Domain-Driven on every metric despite having only one extra predictor, confirming the log transformation and interaction term add genuine explanatory value.
- The gap between Full Model and the other two is meaningful — including ts_pct and age clearly improves prediction.

## Final Model Selection — Model 3 (Transformed)

Despite the Full Model achieving the best raw metrics, **Model 3 (Transformed) is selected as the final model** for the following three reasons:

**1. Statistical reason — multicollinearity:**
The Full Model has a condition number of 1,140, indicating serious multicollinearity between predictors (particularly pts, usg_pct, and ts_pct). This means individual coefficients in Model 1 are unreliable — their standard errors are inflated and their values shift depending on which other predictors are included. Model 3 has a condition number of only 242, meaning its coefficients are stable and trustworthy. A model with reliable coefficients is more useful than one with a marginally better R².

**2. Theoretical reason — log transformation:**
Model 3 applies log(pts + 1) to capture diminishing returns to scoring. This is motivated by basketball economics: the performance gain from scoring 5 to 15 points per game is far greater than the gain from 25 to 35. A raw linear model treats these equally, which is theoretically incorrect. The interaction term (pts × usg_pct) further captures that high scoring only benefits the team when paired with proportional usage — a finding that has no equivalent in Models 1 or 2.

**3. Practical reason — interpretability and generalisability:**
Model 3 achieves an RMSE of 6.016 compared to Model 1's 5.822 — a difference of only 0.194 net_rating points. This marginal improvement in prediction accuracy does not justify the multicollinearity problems and loss of coefficient reliability in Model 1. Model 3 is cleaner, more interpretable, and its coefficients generalise better to unseen data.

**Conclusion:** Model 3 (Transformed) is selected as the final model. It balances statistical soundness, theoretical motivation, and practical performance. All subsequent analysis — sensitivity analysis and stratified modelling — will be conducted on Model 3 only.